# Dataset unificado — Clusterização de municípios brasileiros por perfil socioeconômico

Versão atualizada do `dataset_.ipynb`: mantém a estrutura original (`limpar_df` → merges → `df_final` → Excel) e acrescenta todas as variáveis do estudo, **todas no nível municipal** e validadas contra as APIs.

| Dimensão | Variáveis | Fonte | Ano |
|---|---|---|---|
| Demografia | população, área, densidade, idade mediana, índice de envelhecimento, % urbana | Censo 2022 (T4714, T9515, T9923) + estimativa (T6579) | 2022/2025 |
| Educação | taxa de alfabetização 15+ | Censo 2022 (T9543) | 2022 |
| Saneamento | % domicílios: água rede, esgoto adequado, lixo coletado | Censo 2022 (T6803/6805/6892) | 2022 |
| Renda | rendimento domiciliar per capita | Censo 2022 (T10295) | 2022 |
| Pobreza | % moradores com renda pc até 1/2 SM | Censo 2022 (T10296) | 2022 |
| Economia | PIB, PIB per capita, participação setorial no VAB | T5938 | 2023/2021* |
| Saúde | taxa de mortalidade infantil trienal | Registro Civil (T2654, T2609) | 2022–2024 |
| Desenvolvimento | IDHM, esperança de vida, Gini, renda pc, mort. infantil, analfabetismo, % pobres | IPEADATA/Atlas DH | 2010** |
| Vulnerabilidade | % pop. Bolsa Família, famílias CadÚnico | MDS/SAGI (MI Social) | mensal mais recente |
| Educação (opcional) | IDEB anos iniciais/finais | INEP via Base dos Dados | 2023 |

\* Os VABs setoriais têm defasagem maior que o PIB; o código usa o ano mais recente disponível. \*\* Séries censitárias do Atlas (Gini municipal de 2022 ainda não existe no SIDRA). Correções em relação à versão anterior: a var 37 da T5938 é o **PIB total** (o per capita agora é calculado); a renda per capita municipal do Censo 2022 é a **T10295 var 13431** (a T7435 não é a tabela adequada); a idade mediana está na **T9515 var 10613** (a T9514 não contém essa variável).

In [1]:
!pip install -q sidrapy requests pandas openpyxl pyarrow

In [2]:
import os
import time
import requests
import pandas as pd
import sidrapy

pd.set_option("display.max_columns", 60)
os.makedirs("dados_brutos", exist_ok=True)


def limpar_df(df, nome_valor):
    """Padroniza a saída do sidrapy: cabeçalho na linha 0, código IBGE e valor."""
    df.columns = df.iloc[0].values
    df = df[1:].copy()
    df = df[["Município (Código)", "Valor"]]
    df.columns = ["codigo_ibge", nome_valor]
    # Convenção SIDRA: '-' = zero; '...' = não disponível; '..' = não se aplica; 'X' = sigiloso
    df[nome_valor] = pd.to_numeric(df[nome_valor].replace({"-": "0"}), errors="coerce")
    df["codigo_ibge"] = df["codigo_ibge"].astype(str)
    return df


def baixar_sidra(tabela, variavel, periodo, nome_valor, classificacoes=None):
    """Extrai uma variável de uma tabela SIDRA para todos os municípios e já limpa."""
    bruto = sidrapy.get_table(
        table_code=str(tabela), territorial_level="6", ibge_territorial_code="all",
        period=str(periodo), variable=str(variavel), classifications=classificacoes,
    )
    bruto.to_csv(f"dados_brutos/sidra_t{tabela}_v{variavel}.csv", index=False)
    return limpar_df(bruto, nome_valor)

## 1. Malha de referência (5.571 municípios, com UF e região)

In [3]:
resp = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/municipios", timeout=120).json()
df_final = pd.DataFrame([{
    "codigo_ibge": str(m["id"]),
    "municipio": m["nome"],
    "uf": m["microrregiao"]["mesorregiao"]["UF"]["sigla"] if m.get("microrregiao") else m["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"],
    "regiao": m["microrregiao"]["mesorregiao"]["UF"]["regiao"]["sigla"] if m.get("microrregiao") else m["regiao-imediata"]["regiao-intermediaria"]["UF"]["regiao"]["sigla"],
} for m in resp])
print(df_final.shape)
df_final.head(3)

(5571, 4)


,codigo_ibge,municipio,uf,regiao
0,1100015,Alta Floresta D'Oeste,RO,N
1,1100023,Ariquemes,RO,N
2,1100031,Cabixi,RO,N


## 2. Demografia — população, área, densidade (T4714), idade mediana e envelhecimento (T9515), % urbana (T9923), estimativa 2025 (T6579)

In [4]:
print("Baixando bloco demográfico...")
demografia = [
    baixar_sidra(4714, 93,    2022, "populacao_2022"),
    baixar_sidra(4714, 6318,  2022, "area_km2"),
    baixar_sidra(4714, 614,   2022, "densidade_hab_km2"),
    baixar_sidra(9515, 10613, 2022, "idade_mediana"),
    baixar_sidra(9515, 10612, 2022, "indice_envelhecimento"),
    baixar_sidra(9923, 1000093, 2022, "pct_pop_urbana", classificacoes={"1": "1"}),  # situação = Urbana
    baixar_sidra(6579, 9324,  2025, "pop_estimada_2025"),
]
for d in demografia:
    df_final = df_final.merge(d, on="codigo_ibge", how="left")
    time.sleep(1)
print(df_final.shape)
df_final.head(3)

Baixando bloco demográfico...
(5571, 11)


,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664


## 3. Educação — taxa de alfabetização 15+ (T9543)

In [5]:
df_final = df_final.merge(
    baixar_sidra(9543, 2513, 2022, "tx_alfabetizacao_15mais",
                 classificacoes={"2": "6794", "86": "95251", "287": "100362"}),  # totais
    on="codigo_ibge", how="left")
df_final.head(3)

,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82


## 4. Saneamento — Censo 2022 (T6803, T6805, T6892)
Percentual de domicílios com água da rede geral (forma principal), esgoto por rede geral/pluvial/fossa ligada e lixo coletado.

In [6]:
saneamento = {
    "pct_dom_agua_rede":     (6803, {"1821": "72144"}),
    "pct_dom_esgoto_rede":   (6805, {"11558": "46290"}),
    "pct_dom_lixo_coletado": (6892, {"67": "2520"}),
}
for nome, (tab, cla) in saneamento.items():
    df_final = df_final.merge(
        baixar_sidra(tab, 1000381, 2022, nome, classificacoes=cla),
        on="codigo_ibge", how="left")
    time.sleep(1)
df_final.head(3)

,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06


## 5. Renda — rendimento domiciliar per capita, Censo 2022 (T10295, var 13431)

In [7]:
df_final = df_final.merge(
    baixar_sidra(10295, 13431, 2022, "renda_dom_pc_2022",
                 classificacoes={"2": "6794", "86": "95251", "58": "95253"}),  # totais
    on="codigo_ibge", how="left")
df_final.head(3)

,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado,renda_dom_pc_2022
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48,1210.60
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42,1458.57
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06,1323.79


## 5b. Pobreza monetária — Censo 2022 (T10296)
Percentual de moradores em domicílios com rendimento domiciliar per capita **até 1/2 salário mínimo** (soma das classes "até 1/4 SM" e "1/4 a 1/2 SM"). Meio salário mínimo per capita é o corte de baixa renda do CadÚnico, o que torna esse o proxy censitário padrão de pobreza. Substitui, com dado de 2022, o papel do `pct_pobres_2010` do Atlas.

In [8]:
bruto = sidrapy.get_table(
    table_code="10296", territorial_level="6", ibge_territorial_code="all",
    period="2022", variable="1013604",                      # % do total de moradores
    classifications={"386": "9681,9682", "2": "6794", "86": "95251"},  # até 1/4 SM + 1/4 a 1/2 SM
).iloc[1:]
bruto.to_csv("dados_brutos/sidra_t10296_pobreza.csv", index=False)
bruto["V"] = pd.to_numeric(bruto["V"].replace({"-": "0"}), errors="coerce")
pobreza = bruto.groupby("D1C", as_index=False)["V"].sum()
pobreza.columns = ["codigo_ibge", "pct_pop_ate_meio_sm_2022"]
pobreza["codigo_ibge"] = pobreza["codigo_ibge"].astype(str)

df_final = df_final.merge(pobreza, on="codigo_ibge", how="left")
print("mediana nacional (% até 1/2 SM):", round(df_final["pct_pop_ate_meio_sm_2022"].median(), 1))
df_final.head(3)

mediana nacional (% até 1/2 SM): 30.6


,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado,renda_dom_pc_2022,pct_pop_ate_meio_sm_2022
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48,1210.60,27.52
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42,1458.57,23.80
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06,1323.79,22.88


## 6. PIB e estrutura produtiva (T5938)
A var 37 é o **PIB total** (R$ mil). O per capita é calculado com a população do Censo 2022. Os VABs setoriais são divulgados com defasagem: o código detecta o ano mais recente com dados.

In [9]:
vars_pib = {
    "pib_mil_reais":        37,
    "vab_agropecuaria":     513,
    "vab_industria":        517,
    "vab_servicos_exc_adm": 6575,
    "vab_adm_publica":      525,
}

def pib_ultimo_ano(variavel, nome_valor, periodos="2021,2022,2023"):
    bruto = sidrapy.get_table(table_code="5938", territorial_level="6",
                              ibge_territorial_code="all", period=periodos,
                              variable=str(variavel)).iloc[1:]
    bruto["V"] = pd.to_numeric(bruto["V"].replace({"-": "0"}), errors="coerce")
    piv = bruto.pivot_table(index="D1C", columns="D2N", values="V", aggfunc="first")
    ano = [a for a in sorted(piv.columns) if piv[a].notna().sum() > 0][-1]
    print(f"  {nome_valor}: ano {ano}")
    out = piv[[ano]].reset_index()
    out.columns = ["codigo_ibge", nome_valor]
    out["codigo_ibge"] = out["codigo_ibge"].astype(str)
    return out

for nome, v in vars_pib.items():
    df_final = df_final.merge(pib_ultimo_ano(v, nome), on="codigo_ibge", how="left")
    time.sleep(1)

df_final["pib_per_capita"] = df_final["pib_mil_reais"] * 1000 / df_final["populacao_2022"]
vab_total = df_final[["vab_agropecuaria","vab_industria","vab_servicos_exc_adm","vab_adm_publica"]].sum(axis=1)
for setor in ["agropecuaria", "industria", "servicos_exc_adm", "adm_publica"]:
    df_final[f"share_vab_{setor}"] = 100 * df_final[f"vab_{setor}"] / vab_total
df_final.head(3)

  pib_mil_reais: ano 2023
  vab_agropecuaria: ano 2021
  vab_industria: ano 2021
  vab_servicos_exc_adm: ano 2021
  vab_adm_publica: ano 2021


,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado,renda_dom_pc_2022,pct_pop_ate_meio_sm_2022,pib_mil_reais,vab_agropecuaria,vab_industria,vab_servicos_exc_adm,vab_adm_publica,pib_per_capita,share_vab_agropecuaria,share_vab_industria,share_vab_servicos_exc_adm,share_vab_adm_publica
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48,1210.60,27.52,1046343.0,311469.0,27845.0,172435.0,173122.0,48680.701591,45.478492,4.065729,25.177734,25.278045
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42,1458.57,23.80,4383815.0,293001.0,407675.0,1307977.0,782306.0,45271.911435,10.498219,14.606986,46.864787,28.030007
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06,1323.79,22.88,300832.0,136659.0,8117.0,35866.0,46579.0,56219.772005,60.143649,3.572293,15.784633,20.499426


## 7. Mortalidade infantil trienal 2022–2024 — Registro Civil (T2654 e T2609)
Óbitos de menores de 1 ano ÷ nascidos vivos × 1.000, somados no triênio para estabilizar municípios pequenos. O Registro Civil tem leve sub-registro em relação ao SIM/SINASC, mas é oficial, atual (até 2024) e dispensa credenciais.

In [10]:
def sidra_soma_anos(tabela, variavel, classificacoes, nome_valor, periodos="2022,2023,2024"):
    bruto = sidrapy.get_table(table_code=str(tabela), territorial_level="6",
                              ibge_territorial_code="all", period=periodos,
                              variable=str(variavel), classifications=classificacoes).iloc[1:]
    bruto["V"] = pd.to_numeric(bruto["V"].replace({"-": "0"}), errors="coerce")
    soma = bruto.groupby("D1C", as_index=False)["V"].sum()
    soma.columns = ["codigo_ibge", nome_valor]
    soma["codigo_ibge"] = soma["codigo_ibge"].astype(str)
    return soma

obitos = sidra_soma_anos(2654, 343, {"244": "0", "1836": "0", "2": "0", "260": "5922", "257": "0"},
                         "obitos_menor1ano")
time.sleep(1)
nascidos = sidra_soma_anos(2609, 217, {"232": "0", "240": "0", "2": "0"}, "nascidos_vivos")

tmi = obitos.merge(nascidos, on="codigo_ibge")
tmi["tmi_2022_2024"] = 1000 * tmi["obitos_menor1ano"] / tmi["nascidos_vivos"].replace({0: pd.NA})
df_final = df_final.merge(tmi[["codigo_ibge", "tmi_2022_2024"]], on="codigo_ibge", how="left")
print("TMI mediana (por mil):", round(df_final["tmi_2022_2024"].median(), 2))
df_final.head(3)

TMI mediana (por mil): 9.66


,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado,renda_dom_pc_2022,pct_pop_ate_meio_sm_2022,pib_mil_reais,vab_agropecuaria,vab_industria,vab_servicos_exc_adm,vab_adm_publica,pib_per_capita,share_vab_agropecuaria,share_vab_industria,share_vab_servicos_exc_adm,share_vab_adm_publica,tmi_2022_2024
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48,1210.60,27.52,1046343.0,311469.0,27845.0,172435.0,173122.0,48680.701591,45.478492,4.065729,25.177734,25.278045,9.469697
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42,1458.57,23.80,4383815.0,293001.0,407675.0,1307977.0,782306.0,45271.911435,10.498219,14.606986,46.864787,28.030007,17.281106
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06,1323.79,22.88,300832.0,136659.0,8117.0,35866.0,46579.0,56219.772005,60.143649,3.572293,15.784633,20.499426,0.000000


## 8. Atlas do Desenvolvimento Humano — IPEADATA (Censo 2010)
IDHM, esperança de vida, Gini, renda per capita, mortalidade infantil, analfabetismo e % de pobres. São de 2010 (censitárias); a persistência dos padrões territoriais justifica o uso — documente a defasagem na metodologia.

In [11]:
SERIES_ADH = {
    "ADH_IDHM":       "idhm_2010",
    "ADH_ESPVIDA":    "esperanca_vida_2010",
    "ADH_GINI":       "gini_2010",
    "ADH_RDPC":       "renda_pc_2010",
    "ADH_MORT1":      "mortalidade_infantil_2010",
    "ADH_T_ANALF15M": "tx_analfabetismo_2010",
    "ADH_PMPOB":      "pct_pobres_2010",
}

def buscar_adh(codigo_serie, nome_coluna, ano=2010, tentativas=4):
    """A API do IPEADATA oscila; tenta novamente com espera crescente."""
    url = f"http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='{codigo_serie}')"
    for t in range(tentativas):
        try:
            df = pd.DataFrame(requests.get(url, timeout=180).json()["value"])
            break
        except Exception as e:
            if t == tentativas - 1:
                raise
            print(f"    (tentativa {t+1} falhou para {codigo_serie}, aguardando...)")
            time.sleep(10 * (t + 1))
    df = df[df["NIVNOME"] == "Municípios"].copy()
    df["ano"] = pd.to_datetime(df["VALDATA"], utc=True).dt.year
    df = df[df["ano"] == ano][["TERCODIGO", "VALVALOR"]]
    df.columns = ["codigo_ibge", nome_coluna]
    df["codigo_ibge"] = df["codigo_ibge"].astype(str)
    return df

for cod, nome in SERIES_ADH.items():
    d = buscar_adh(cod, nome)
    d.to_csv(f"dados_brutos/ipea_{cod}.csv", index=False)
    df_final = df_final.merge(d, on="codigo_ibge", how="left")
    print(f"  {cod}: {len(d)} municípios")
    time.sleep(2)
df_final.head(3)

  ADH_IDHM: 5564 municípios
  ADH_ESPVIDA: 5564 municípios
  ADH_GINI: 5564 municípios
  ADH_RDPC: 5564 municípios
  ADH_MORT1: 5564 municípios
  ADH_T_ANALF15M: 5564 municípios
  ADH_PMPOB: 5564 municípios


,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado,renda_dom_pc_2022,pct_pop_ate_meio_sm_2022,pib_mil_reais,vab_agropecuaria,vab_industria,vab_servicos_exc_adm,vab_adm_publica,pib_per_capita,share_vab_agropecuaria,share_vab_industria,share_vab_servicos_exc_adm,share_vab_adm_publica,tmi_2022_2024,idhm_2010,esperanca_vida_2010,gini_2010,renda_pc_2010,mortalidade_infantil_2010,tx_analfabetismo_2010,pct_pobres_2010
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48,1210.60,27.52,1046343.0,311469.0,27845.0,172435.0,173122.0,48680.701591,45.478492,4.065729,25.177734,25.278045,9.469697,0.641,70.75,0.58,476.99,23.8,11.99,26.04
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42,1458.57,23.80,4383815.0,293001.0,407675.0,1307977.0,782306.0,45271.911435,10.498219,14.606986,46.864787,28.030007,17.281106,0.702,73.36,0.53,689.95,19.2,7.90,11.54
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06,1323.79,22.88,300832.0,136659.0,8117.0,35866.0,46579.0,56219.772005,60.143649,3.572293,15.784633,20.499426,0.000000,0.650,70.39,0.51,457.17,24.8,13.63,21.20


## 9. Bolsa Família e CadÚnico — API MI Social (SAGI/MDS)
Competência mensal mais recente com dados do PBF (detectada automaticamente). O `codigo_ibge` do MDS tem 6 dígitos; o merge usa os 6 primeiros dígitos do código de 7.

In [12]:
URL_MISOCIAL = "https://aplicacoes.mds.gov.br/sagi/servicos/misocial"

r = requests.get(URL_MISOCIAL, params={
    "q": "*", "fq": "qtd_familias_beneficiarias_bolsa_familia_i:[0 TO *]",
    "wt": "json", "rows": 1, "sort": "anomes desc"}, timeout=60)
ANOMES_PBF = r.json()["response"]["docs"][0]["anomes"]
print("Competência utilizada:", ANOMES_PBF)

campos = ("codigo_ibge,qtd_familias_beneficiarias_bolsa_familia_i,"
          "cadun_qtd_familias_cadastradas_i,pbf_media_pessoas_benef_f")
r = requests.get(URL_MISOCIAL, params={
    "q": "*", "fq": [f"anomes_s:{ANOMES_PBF}", "qtd_familias_beneficiarias_bolsa_familia_i:[0 TO *]"],
    "wt": "json", "rows": 6000, "fl": campos}, timeout=180)
pbf = pd.DataFrame(r.json()["response"]["docs"]).rename(columns={
    "qtd_familias_beneficiarias_bolsa_familia_i": "fam_bolsa_familia",
    "cadun_qtd_familias_cadastradas_i": "fam_cadunico",
    "pbf_media_pessoas_benef_f": "media_pessoas_fam_pbf",
})
pbf.to_csv(f"dados_brutos/mds_misocial_{ANOMES_PBF}.csv", index=False)
pbf["cod_ibge6"] = pbf["codigo_ibge"].astype(str)
pbf["pessoas_pbf_estim"] = pbf["fam_bolsa_familia"] * pbf["media_pessoas_fam_pbf"]

df_final["cod_ibge6"] = df_final["codigo_ibge"].str[:6]
df_final = df_final.merge(pbf[["cod_ibge6", "fam_bolsa_familia", "fam_cadunico", "pessoas_pbf_estim"]],
                          on="cod_ibge6", how="left").drop(columns=["cod_ibge6"])
df_final["pct_pop_bolsa_familia"] = 100 * df_final["pessoas_pbf_estim"] / df_final["populacao_2022"]
df_final["fam_cadunico_por_100hab"] = 100 * df_final["fam_cadunico"] / df_final["populacao_2022"]
df_final = df_final.drop(columns=["pessoas_pbf_estim"])
df_final.head(3)

Competência utilizada: 202606


,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado,renda_dom_pc_2022,pct_pop_ate_meio_sm_2022,pib_mil_reais,vab_agropecuaria,vab_industria,vab_servicos_exc_adm,vab_adm_publica,pib_per_capita,share_vab_agropecuaria,share_vab_industria,share_vab_servicos_exc_adm,share_vab_adm_publica,tmi_2022_2024,idhm_2010,esperanca_vida_2010,gini_2010,renda_pc_2010,mortalidade_infantil_2010,tx_analfabetismo_2010,pct_pobres_2010,fam_bolsa_familia,fam_cadunico,pct_pop_bolsa_familia,fam_cadunico_por_100hab
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48,1210.60,27.52,1046343.0,311469.0,27845.0,172435.0,173122.0,48680.701591,45.478492,4.065729,25.177734,25.278045,9.469697,0.641,70.75,0.58,476.99,23.8,11.99,26.04,1480,4950,21.089606,23.029683
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42,1458.57,23.80,4383815.0,293001.0,407675.0,1307977.0,782306.0,45271.911435,10.498219,14.606986,46.864787,28.030007,17.281106,0.702,73.36,0.53,689.95,19.2,7.90,11.54,7697,22120,21.262379,22.843452
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06,1323.79,22.88,300832.0,136659.0,8117.0,35866.0,46579.0,56219.772005,60.143649,3.572293,15.784633,20.499426,0.000000,0.650,70.39,0.51,457.17,24.8,13.63,21.20,340,1296,19.304802,24.219772


In [15]:
!pip install basedosdados

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 24.5 MB/s eta 0:00:00


## 10. IDEB (opcional)
**Opção A** — `basedosdados` (requer projeto de faturamento no Google Cloud). **Opção B** — baixe manualmente os zips municipais em [gov.br/inep](https://www.gov.br/inep/pt-br/areas-de-atuacao/pesquisas-estatisticas-e-indicadores/ideb/resultados) (o INEP bloqueia downloads automatizados) e aponte o caminho do `.xlsx`.

In [17]:
EXECUTAR_IDEB_BD = True        # Opção A: mude para True com o basedosdados configurado
BILLING_ID = "interpretador-de-contas"
#CAMINHO_IDEB_XLSX = None        # Opção B: ex. "divulgacao_anos_iniciais_municipios_2023.xlsx"

if EXECUTAR_IDEB_BD:
    import basedosdados as bd
    ideb = bd.read_sql("""
        SELECT id_municipio, anos_escolares, ideb
        FROM `basedosdados.br_inep_ideb.municipio`
        WHERE ano = 2023 AND rede = 'publica'
    """, billing_project_id=BILLING_ID)
    ideb = (ideb.pivot_table(index="id_municipio", columns="anos_escolares", values="ideb", aggfunc="mean")
                .rename(columns={"iniciais (1-5)": "ideb_anos_iniciais", "finais (6-9)": "ideb_anos_finais"})
                .reset_index().rename(columns={"id_municipio": "codigo_ibge"}))
    ideb["codigo_ibge"] = ideb["codigo_ibge"].astype(str)
    df_final = df_final.merge(ideb, on="codigo_ibge", how="left")
elif CAMINHO_IDEB_XLSX:
    raw = pd.read_excel(CAMINHO_IDEB_XLSX, skiprows=9)
    col_ideb = [c for c in raw.columns if "IDEB" in str(c).upper()][-1]
    ideb = (raw[raw["REDE"].str.contains("blica", na=False)][["CO_MUNICIPIO", col_ideb]]
            .rename(columns={"CO_MUNICIPIO": "codigo_ibge", col_ideb: "ideb_anos_iniciais"}))
    ideb["codigo_ibge"] = ideb["codigo_ibge"].astype(int).astype(str)
    ideb["ideb_anos_iniciais"] = pd.to_numeric(ideb["ideb_anos_iniciais"], errors="coerce")
    df_final = df_final.merge(ideb, on="codigo_ibge", how="left")
else:
    print("IDEB não incluído (configure a Opção A ou B acima).")

Downloading: 100%|██████████|


## 11. Diagnóstico e dataset final

In [18]:
print("Dimensão final:", df_final.shape)
faltantes = df_final.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
print("\nValores ausentes por coluna:")
print(faltantes.to_string() if len(faltantes) else "  nenhum!")

# Municípios sem Atlas 2010 = criados após 2010; sem Censo = município novo de 2025
df_final[df_final["idhm_2010"].isna()][["codigo_ibge", "municipio", "uf"]]

Dimensão final: (5571, 42)

Valores ausentes por coluna:
todos (1-4)                   506
ideb_anos_finais              189
ideb_anos_iniciais            141
pct_pobres_2010                 7
tx_analfabetismo_2010           7
gini_2010                       7
idhm_2010                       7
renda_pc_2010                   7
esperanca_vida_2010             7
mortalidade_infantil_2010       7
populacao_2022                  1
densidade_hab_km2               1
area_km2                        1
idade_mediana                   1
indice_envelhecimento           1
pct_dom_agua_rede               1
pct_dom_esgoto_rede             1
pct_pop_urbana                  1
tx_alfabetizacao_15mais         1
share_vab_agropecuaria          1
pib_per_capita                  1
vab_adm_publica                 1
vab_servicos_exc_adm            1
vab_industria                   1
vab_agropecuaria                1
pib_mil_reais                   1
pct_pop_ate_meio_sm_2022        1
renda_dom_pc_2022        

,codigo_ibge,municipio,uf
224,1504752,Mojuí dos Campos,PA
803,2206720,Nazária,PI
4503,4212650,Pescaria Brava,SC
4605,4220000,Balneário Rincão,SC
4923,4314548,Pinto Bandeira,RS
5160,5006275,Paraíso das Águas,MS
5199,5101837,Boa Esperança do Norte,MT


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
# Salva o dataset unificado
df_final.to_excel("/content/drive/MyDrive/Trabalho2AM/dados/dados_municipais_completos.xlsx", index=False)
df_final.to_csv("/content/drive/MyDrive/Trabalho2AM/dados/dados_municipais_completos.csv", index=False)
df_final.to_parquet("/content/drive/MyDrive/Trabalho2AM/dados/dados_municipais_completos.parquet", index=False)
print(f"Sucesso! Dataset com {df_final.shape[0]} municípios e {df_final.shape[1]} variáveis salvo em xlsx/csv/parquet.")
df_final.head()

Sucesso! Dataset com 5571 municípios e 42 variáveis salvo em xlsx/csv/parquet.


,codigo_ibge,municipio,uf,regiao,populacao_2022,area_km2,densidade_hab_km2,idade_mediana,indice_envelhecimento,pct_pop_urbana,pop_estimada_2025,tx_alfabetizacao_15mais,pct_dom_agua_rede,pct_dom_esgoto_rede,pct_dom_lixo_coletado,renda_dom_pc_2022,pct_pop_ate_meio_sm_2022,pib_mil_reais,vab_agropecuaria,vab_industria,vab_servicos_exc_adm,vab_adm_publica,pib_per_capita,share_vab_agropecuaria,share_vab_industria,share_vab_servicos_exc_adm,share_vab_adm_publica,tmi_2022_2024,idhm_2010,esperanca_vida_2010,gini_2010,renda_pc_2010,mortalidade_infantil_2010,tx_analfabetismo_2010,pct_pobres_2010,fam_bolsa_familia,fam_cadunico,pct_pop_bolsa_familia,fam_cadunico_por_100hab,ideb_anos_finais,ideb_anos_iniciais,todos (1-4)
0,1100015,Alta Floresta D'Oeste,RO,N,21494.0,7067.127,3.04,34.0,44.61,60.35,22787,91.59,26.48,0.47,61.48,1210.60,27.52,1046343.0,311469.0,27845.0,172435.0,173122.0,48680.701591,45.478492,4.065729,25.177734,25.278045,9.469697,0.641,70.75,0.58,476.99,23.8,11.99,26.04,1480,4950,21.089606,23.029683,4.7,5.3,4.5
1,1100023,Ariquemes,RO,N,96833.0,4426.571,21.88,32.0,34.61,86.70,109170,94.08,54.34,1.99,90.42,1458.57,23.80,4383815.0,293001.0,407675.0,1307977.0,782306.0,45271.911435,10.498219,14.606986,46.864787,28.030007,17.281106,0.702,73.36,0.53,689.95,19.2,7.90,11.54,7697,22120,21.262379,22.843452,4.8,5.4,4.2
2,1100031,Cabixi,RO,N,5351.0,1314.352,4.07,38.0,64.14,53.19,5664,89.82,39.30,1.58,61.06,1323.79,22.88,300832.0,136659.0,8117.0,35866.0,46579.0,56219.772005,60.143649,3.572293,15.784633,20.499426,0.000000,0.650,70.39,0.51,457.17,24.8,13.63,21.20,340,1296,19.304802,24.219772,5.0,5.8,4.1
3,1100049,Cacoal,RO,N,86887.0,3793.000,22.91,33.0,43.50,82.76,98280,93.71,80.07,67.19,85.66,1625.88,21.03,3848468.0,341315.0,236801.0,1200654.0,629109.0,44292.794089,14.174923,9.834423,49.863552,26.127102,8.581524,0.718,74.27,0.57,738.06,14.3,8.29,13.08,5591,18195,17.766754,20.940992,4.9,5.7,4.2
4,1100056,Cerejeiras,RO,N,15890.0,2783.300,5.71,34.0,47.01,88.84,16966,92.15,49.48,57.99,87.04,1554.33,19.72,1006389.0,151690.0,27433.0,308668.0,123365.0,63334.738829,24.820177,4.488707,50.505599,20.185517,5.069708,0.692,72.94,0.50,577.18,18.1,10.29,13.70,1068,3409,19.156703,21.453744,5.0,5.8,4.1
